# Lesson 60 · How autograd works — checking your answers against PyTorch: the `grad_fn` graph vs your hand-written `Value` engine

**Goal**: compute the same thing two ways — PyTorch's `grad_fn` and your hand-written `Value` must arrive at the same gradients.

> **A verification lesson**: line the answers up first, and then you can hand `loss.backward()` over to PyTorch with confidence.

🔗 **Aurora connection**: every model you train in Months 2–5 will call `.backward()` again and again. This lesson nails down the mechanism underneath, so that when torch later replaces the hand-written `Value` graph, you know you aren't swapping in a black box — it's the same mathematics. The concrete entry point: every `loss.backward()` in the training loop of the Aurora ML engine (`src/aurora/ml/`, a planned module) relies on this mechanism.

← **Previous lesson** · [L59 · PyTorch tensors](L59_tensor_basics.ipynb)

> Last lesson set up **PyTorch tensors**: converting to and from NumPy, devices, `requires_grad`.
> This lesson moves on to see how **autograd** keeps the books for you.

## This lesson's story: how does PyTorch "remember" what operations you did?

In L54–L56 we implemented a `Value` class ourselves — every node carried a `_backward` closure and a `_prev` set, like a trail map sketched by hand on paper. PyTorch does exactly the same thing; it just makes the map bigger, faster, and a lot less work.

**The core mechanism**: every Tensor with `requires_grad=True` automatically gets a `grad_fn` (gradient function) attached — think of it as a little sticky note saying "here's where I came from", so backpropagation can retrace its steps:

```python
a = torch.tensor(2.0, requires_grad=True)   # leaf node, grad_fn=None
b = torch.tensor(3.0, requires_grad=True)   # leaf node
c = a * b                                    # c.grad_fn = MulBackward0
d = c + a                                    # d.grad_fn = AddBackward0
```

The graph looks like this:
```
a ──┬──[×]──► c ──[+]──► d
b ──┘         a ──────────┘
```

**Leaf vs non-leaf nodes**:

| Type | Definition | `.grad` rule |
|---|---|---|
| Leaf node | `requires_grad=True` and not the result of an operation | `.grad` filled in automatically after backward |
| Non-leaf node | created by an operation | `.grad=None` (unless you call `retain_grad()`) |

**Why non-leaf nodes don't store gradients by default**: there can be millions of intermediate activations, and storing all of them would blow through GPU memory. What training really needs to keep around is usually just the gradients of the parameters, so by default PyTorch keeps only the leaf nodes.

This lesson's task: verify the gradients of `L = 2*a*b + 3*b` with respect to `a, b`, and understand `detach()`, `retain_grad()`, and gradient accumulation.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import torch
import numpy as np

## 1. The computation graph: `requires_grad=True` and leaf / non-leaf nodes

When a tensor has `requires_grad=True`, every operation leaves a `grad_fn` on the graph, recording "how this step was computed".
**Leaf tensor**: created directly by the user, `is_leaf == True`; once backpropagation finishes, its gradient is written into `.grad`.
**Non-leaf node**: derived from an operation, `is_leaf == False`; by default its gradient is not kept (`.grad` is `None`) — if you want to inspect it, call `retain_grad()` first.

```
a (leaf)  b (leaf)
  \          /
   c = a * b   <- c.grad_fn = MulBackward0, c.is_leaf = False
       |
   L = c + a   <- L.grad_fn = AddBackward0
```

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
L = c + a

print('a.is_leaf:', a.is_leaf)   # True
print('c.is_leaf:', c.is_leaf)   # False
print('c.grad_fn:', c.grad_fn)   # MulBackward0
print('L.grad_fn:', L.grad_fn)   # AddBackward0

### Going deeper: why split leaf and non-leaf?

You might ask: "Why not just store the gradient of every tensor, so we never have to compute it again?"

**The reality of GPU memory**: a modern deep learning model has millions or even billions of parameters. If you have a 12-layer network with 1 million activations per layer, then just keeping the gradients of those intermediate activations takes 12 × 1 million = 12 million floats — per example. Multiply that by the batch size and by the size of real models, and it can run to tens of GB of GPU memory.

But what we actually update during training are the model's **parameters** (leaf nodes). The gradients of activations (non-leaf nodes) are just intermediate steps that can be thrown away once used. That's exactly why PyTorch doesn't keep non-leaf gradients by default.

**The initial gradient value**: before `backward()`, `.grad` is `None` (not 0). PyTorch only **creates** the `.grad` tensor for a leaf node, and fills in its value, on the first `backward()`. If you run `backward()` again, the new gradient is **added to** the existing `.grad` — that's "gradient accumulation".

**Edge cases**:
```python
a = torch.tensor(2.0, requires_grad=True)
c = a * 2  # c is a non-leaf node (produced by an operation)
d = torch.tensor(3.0, requires_grad=True)  # even though it's created on this line, d is still a leaf
```
The key point: **only "created directly + requires_grad=True" makes a leaf; every operation result counts as a non-leaf**.

**A concrete example**: in a 3-layer MLP, which gradients are kept?
```
input → [Linear 1] → a1 → [ReLU] → b1 → [Linear 2] → a2 → [ReLU] → b2 → [Linear 3] → logit → Loss
  leaf     param        non-leaf       param        non-leaf       param        non-leaf    |
                                                                                              ↓
                                                                                            backward
```
- **Kept**: the weights and biases of the 3 Linear layers (parameters, leaf nodes)
- **Discarded**: the gradients of a1, b1, a2, b2, logit (all intermediate activations)
- **If you need to see intermediate gradients** (say, to debug vanishing gradients), use `retain_grad()` — at the cost of extra memory



## 2. `.detach()` and the rules for storing `.grad`

`.detach()` returns a new tensor that **shares data with the original but cuts off the gradient flow**. Operations on it are no longer recorded in the computation graph.
Common uses:

1. **Debugging / printing**: turn an intermediate tensor into numpy without touching the gradient chain
   ```python
   y = model(x)  # y is on the graph
   print(y.detach().numpy())  # safe to print; the original gradient chain is untouched
   ```

2. **Pinning one path**: separating the gradient paths of the generator and the discriminator in a GAN
   ```python
   fake = generator(z)
   # the discriminator scores fake, but gradients don't flow back into the generator
   d_loss = discriminator(fake.detach())
   d_loss.backward()  # updates only the discriminator's parameters
   ```

3. **Stopping the gradient**: some parameters shouldn't be updated in this step
   ```python
   # EMA (exponential moving average) update: target_net is frozen
   target_y = target_net(x).detach()
   loss = (model(x) - target_y) ** 2
   loss.backward()  # updates only model, not target_net
   ```

**Compare**: `.grad` is kept automatically only for leaf nodes; for a non-leaf node you must call `retain_grad()` before `backward()`, or its `.grad` stays `None` forever.

**Important**: `detach()` and `retain_grad()` solve different problems:
- `detach()` says "from here on, stop recording gradients"
- `retain_grad()` says "the gradient was recorded but would normally be thrown away — I want to keep this value"

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x * 2          # non-leaf node
y.retain_grad()    # turn on gradient retention for the non-leaf node
L2 = y * 3
L2.backward()

print('x.grad:', x.grad)   # 6.0  (dL2/dx = 3*2)
print('y.grad:', y.grad)   # 3.0  (dL2/dy = 3)

# detach demo
y_np = y.detach().numpy()   # doesn't affect the graph
print('y_np (detach):', y_np)

## 3. Gradient accumulation and `zero_grad()`

Every call to `.backward()` **adds** the new gradient to `.grad` instead of overwriting it.
This is by design: it supports the gradient-accumulation training trick, and it's also a reminder that in a standard training loop you must clear the gradients first — otherwise old gradients quietly sneak in.

The standard pattern:
```python
for batch in dataloader:
    optimizer.zero_grad()   # clear what's left over from the last step
    loss = model(batch)
    loss.backward()
    optimizer.step()
```

The same goes for the hand-written `Value` engine: before each backward pass you have to reset every node's `.grad` to zero by hand.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

# two backward passes, no clearing
for i in range(2):
    L3 = w * 3.0
    L3.backward()
    print(f'after backward #{i+1}, w.grad = {w.grad}')  # 3.0, 6.0

# clear, then do it again
w.grad.zero_()
L3 = w * 3.0
L3.backward()
print('after zero_grad, w.grad =', w.grad)  # 3.0

### Gradient accumulation in practice: training in chunks when GPU memory runs short

The most common reason gradient accumulation is "by design" is to cope with GPU memory limits.

**Scenario**: suppose we're training a big model and want batch_size=128, but the GPU only has room for batch_size=32.

**Without gradient accumulation** (wrong):
```python
for batch in dataloader:
    loss = model(batch)
    loss.backward()
    optimizer.step()
    # here batch_size is only 32; the gradient averages 32 samples, not 128
```

**With gradient accumulation** (right):
```python
for batch_idx, batch in enumerate(dataloader):
    loss = model(batch)
    loss.backward()  # the gradient is added onto .grad

    if (batch_idx + 1) % 4 == 0:  # accumulate 4 batches (4×32=128)
        optimizer.step()
        optimizer.zero_grad()  # clear, ready for the next round of accumulation
```

This way each parameter update still sees the gradient of 128 samples, while GPU memory only pays for 32 at a time. (If `loss` is a per-batch mean, divide it by 4 before `backward()` so the accumulated gradient is a true 128-sample average rather than 4 times one.) It's a textbook trade-off between memory and how you compute.

**Why the standard training loop needs `zero_grad()`**: if you're not accumulating on purpose, you must clear before every update — otherwise old and new gradients mix and the model learns the wrong thing.

## 3.5 `retain_graph=True` and `retain_grad()`: two orthogonal APIs

These two APIs have very similar names, but what they keep, and when, is completely different. To tell them apart, first look at the three phases of a PyTorch backward pass:

| Phase | When it happens | What gets saved |
|---|---|---|
| **Forward pass** | after `z = w * x` | the `saved_tensors` inside `grad_fn` (e.g. the values of w and x, needed for the derivative) |
| **Backward pass** | while `loss.backward()` runs | the gradient of each node, computed one by one and stored in its `.grad` attribute |
| **Cleanup** | after `backward()` returns | `saved_tensors` are freed by default (to save memory); non-leaf `.grad` values are discarded |

**What `retain_grad()` does**: tells PyTorch "after backpropagation, **keep this non-leaf node's .grad value**".
- **When** it applies: must be called before `backward()`
- **What** it keeps: that node's `.grad` tensor (the gradient value itself)
- **Cost**: tiny (it just doesn't release one tensor reference)

**What `backward(retain_graph=True)` does**: tells PyTorch "after backpropagation, **don't tear down the computation graph yet, and don't free the saved_tensors yet**".
- **When** it applies: specified as an argument to `backward()`
- **What** it keeps: the whole DAG structure of the computation graph plus the `saved_tensors` in each `grad_fn` (including intermediate values saved during the forward pass)
- **Cost**: significant (these saved_tensors can be large — a whole batch of activations, for example)

**What are `saved_tensors`**: taking the derivative on the way back needs intermediate values from the forward pass. For `c = a * b`, to get `∂c/∂a = b`, the backward pass has to know what `b` actually was. During the forward pass PyTorch stashes these "needed again later" values in `grad_fn.saved_tensors`, and by default frees them when backward finishes, to save memory.

**A typical scenario: calling backward on two losses in multi-task learning**

```python
# scenario: one encoder shared by two heads
shared = encoder(x)       # shared.grad_fn = ...
head1_out = head1(shared)  # needs the value of shared to take derivatives
head2_out = head2(shared)  # the same shared

loss1 = criterion(head1_out, y1)
loss2 = criterion(head2_out, y2)

# first backward: keep the graph (loss2 needs it too)
loss1.backward(retain_graph=True)

# second backward: no longer kept, the graph is freed
loss2.backward()
```

If you leave out `retain_graph=True`, the graph is freed after the first backward and the second one raises a RuntimeError.

**Using both APIs together**:
```python
shared = encoder(x)
head1_out = head1(shared)
head2_out = head2(shared)

# suppose shared is a non-leaf node
shared.retain_grad()       # ← keep the value of shared.grad

loss1 = criterion(head1_out, y1)
loss2 = criterion(head2_out, y2)

loss1.backward(retain_graph=True)
print(shared.grad)         # ✅ you can see shared's gradient with respect to loss1

loss2.backward()
print(shared.grad)         # ✅ still visible: now the accumulated gradient from both losses (thanks to retain_grad)
```

**Summary**:
- `retain_grad()` → keeps an intermediate gradient value (for debugging)
- `retain_graph=True` → keeps the computation graph so you can call backward several times (multi-loss setups)
- The two are "orthogonal": one keeps a value, the other keeps the graph; they don't interfere and can be used together

In [ ]:
# === retain_graph=True demo ===

w = torch.tensor(2.0, requires_grad=True)
x_fixed = torch.tensor(3.0)

# one forward pass, two losses
z = w * x_fixed           # z = 6.0
loss1 = z * z             # L1 = z²,  dL1/dw = 2z·x = 36.0
loss2 = z + 1.0           # L2 = z+1, dL2/dw = x   =  3.0

# first backward: keep the graph
loss1.backward(retain_graph=True)
print(f'loss1.backward(retain_graph=True) → w.grad = {w.grad.item()}')
assert abs(w.grad.item() - 36.0) < 1e-6, 'expected dL1/dw = 36.0'

# second backward: not kept any more, the graph is freed here
w.grad.zero_()
loss2.backward()
print(f'loss2.backward()                 → w.grad = {w.grad.item()}')
assert abs(w.grad.item() - 3.0) < 1e-6, 'expected dL2/dw = 3.0'

# compare: without retain_graph the graph is freed early → the second backward raises RuntimeError
print()
w2 = torch.tensor(2.0, requires_grad=True)
z2 = w2 * x_fixed
loss1b = z2 * z2
loss2b = z2 + 1.0
loss1b.backward()  # retain_graph=False (the default): the graph is freed here
try:
    loss2b.backward()
    print('Error: a RuntimeError should have been raised here')
except RuntimeError as e:
    print(f'✅ RuntimeError as expected (graph already freed): {str(e)[:90]}')

print('\nConclusion: retain_graph=True keeps the graph; retain_grad() keeps a non-leaf .grad value. They are orthogonal and can be used together.')



## 4. ✏️ Implement `verify_gradients()`

**Task**: run a minimal example with `L = 2*a*b + 3*b` (`a=2.0, b=3.0`) and confirm that the automatic gradients match the hand-computed ones.

**Hand-computed result**:
```
∂L/∂a = 2b = 6.0
∂L/∂b = 2a + 3 = 7.0
```

**Operator-overloading details** (nothing to worry about):
- For `2 * a`, the Python int 2 doesn't know how to multiply a Tensor, so Python calls `a.__rmul__(2)` (reflected multiplication)
- PyTorch's Tensor implements `__rmul__` and returns `2.0 * a` (still a Tensor)
- `2*a*b` computes `2*a` first (a Tensor), then multiplies by `b` — everything stays on the computation graph
- In the end `L` is a scalar Tensor (shape=[]), so you can call `L.backward()` directly

**Implementation notes**:

| Step | Operation | Watch out |
|---|---|---|
| 1 | `a = torch.tensor(2.0, requires_grad=True)`, same for `b` | must have `requires_grad=True` |
| 2 | `L = 2*a*b + 3*b` | every step is a Tensor operation, so the graph builds itself |
| 3 | `L.backward()` | only a scalar can call backward() directly; a multi-dimensional tensor needs gradient weights passed in |
| 4 | `assert abs(a.grad - 6.0) < 1e-6` | check a's gradient |
| 5 | `assert abs(b.grad - 7.0) < 1e-6` | check b's gradient |
| 6 | `return a.grad.item(), b.grad.item()` | the check cell below verifies the return value |

**Acceptance criteria**: the function runs without errors, returns `(6.0, 7.0)`, and prints `a.grad=6.0, b.grad=7.0`.

### Background: partial derivatives of a multivariable function

Before tackling `verify_gradients()`, a quick refresher on what a partial derivative means.

**What is a partial derivative?** It answers the question: "If I change only one variable and freeze all the others, how does the function change?"

Take `L = 2*a*b + 3*b`:
- `∂L/∂a` = "if I change only a, how does L change? Treat b as a constant"
  - L = 2a·b + 3·b (b frozen)
  - differentiate with respect to a → `∂L/∂a = 2b`

- `∂L/∂b` = "if I change only b, how does L change? Treat a as a constant"
  - L = 2a·b + 3·b (a frozen)
  - differentiate with respect to b → `∂L/∂b = 2a + 3`

**How to compute it**: differentiate with respect to each variable in turn, treating the others as constants
```
L = 2*a*b + 3*b

∂L/∂a: treat b as a constant
  = 2·b + 0  (the derivative of 3*b with respect to a is 0)
  = 2b

∂L/∂b: treat a as a constant
  = 2·a + 3  (2*a is a constant, times the derivative of b, which is 1)
  = 2a + 3
```

**Why does this matter?** In a neural network, the gradients the parameters (weights w, biases b) need are exactly these partial derivatives: `∂loss/∂w`, `∂loss/∂b`.

In [ ]:
def verify_gradients():
    # ✏️ TODO: a=2.0, b=3.0 (requires_grad=True); L = 2*a*b + 3*b; L.backward();
    #          assert both gradients, then return a.grad.item(), b.grad.item()
    raise NotImplementedError("TODO: create a=2.0, b=3.0, L=2*a*b+3*b, backward, assert, then return both gradients")

try:
    print(verify_gradients())
except (NotImplementedError, TypeError) as e:
    print(f'⬜ Not implemented yet: {e}')

In [ ]:
# --- Check: call your verify_gradients() ---
import torch

try:
    result = verify_gradients()
except (NotImplementedError, TypeError):
    result = None

if result is None:
    print("⬜ Implement verify_gradients() first (remember to return a.grad.item(), b.grad.item()), then run this cell")
    print("   Hint: L = 2ab + 3b → dL/da = 2b = 6.0, dL/db = 2a + 3 = 7.0")
else:
    da, db = result
    assert abs(da - 6.0) < 1e-4, f'dL/da should be 6.0 (=2b), got {da}'
    assert abs(db - 7.0) < 1e-4, f'dL/db should be 7.0 (=2a+3), got {db}'
    print(f'✅ dL/da = {da} | dL/db = {db} — matches the analytic values')

## 5. Parameter experiment: the gradient error between home-made Value and torch for `L = tanh(w*x + b)`

**Experiment settings**: sweep `w` over `[-2, -1, 0, 1, 2]`, keep `x=1.0, b=0.5` fixed, and see whether the two sets of gradients line up.

**Chain-rule derivation**:
```
z = w*x + b
L = tanh(z)

∂L/∂w = (∂L/∂z) · (∂z/∂w)
      = (1 - tanh²(z)) · x    ← chain rule

where:
  - ∂z/∂w = x  (the partial derivative of z with respect to w is x)
  - ∂L/∂z = d(tanh)/dz = 1 - tanh²(z)  (the derivative of tanh)
```

**Two common forms of the tanh derivative** (both correct):
1. `1 - tanh²(z)` ← the one we use
2. `sech²(z) = 1/cosh²(z)` ← written with hyperbolic functions

**Why is the derivative of tanh 1 - tanh²?** Derive it from the definition:
```python
# tanh(z) = (e^z - e^-z) / (e^z + e^-z)
# differentiate and simplify: d(tanh)/dz = 1 - tanh²(z)

import math
z = 0.5
tanh_z = math.tanh(z)
derivative_direct = 1 - tanh_z**2
print(f"tanh({z}) = {tanh_z}")
print(f"derivative 1 - tanh²(z) = {derivative_direct}")
# at z=0.5 the derivative ≈ 0.79
```

**What you should see**:
- torch autograd and the hand-written Value agree to within 1e-6
- in tanh's saturation region (large `|w|`, where the output is squashed flat), both gradients head toward 0 together
- the two curves in the plot should overlap almost perfectly, and the error curve should hug the x-axis

In [ ]:
import matplotlib.pyplot as plt
import math

# ---- hand-written Value (minimal implementation, same as L56_backward_pass.ipynb) ----
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _back():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _back
        return out

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _back():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _back
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _back():
            self.grad += (1 - t**2) * out.grad
        out._backward = _back
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()

# ---- comparison experiment ----
ws = [-2.0, -1.0, 0.0, 1.0, 2.0]
x_val, b_val = 1.0, 0.5

grads_torch, grads_value, errors = [], [], []

for w_val in ws:
    # torch
    wt = torch.tensor(w_val, requires_grad=True)
    xt = torch.tensor(x_val)
    bt = torch.tensor(b_val)
    Lt = torch.tanh(wt * xt + bt)
    Lt.backward()
    gt = wt.grad.item()
    grads_torch.append(gt)

    # Value
    wv = Value(w_val)
    xv = Value(x_val)
    bv = Value(b_val)
    Lv = (wv * xv + bv).tanh()
    Lv.backward()
    gv = wv.grad
    grads_value.append(gv)

    errors.append(abs(gt - gv))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(ws, grads_torch, 'o-', label='torch autograd', linewidth=2)
ax1.plot(ws, grads_value, 's--', label='hand-written Value', linewidth=2, alpha=0.7)
ax1.set_xlabel('w')
ax1.set_ylabel('dL/dw')
ax1.set_title('dL/dw for L=tanh(w*x+b)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(ws, [max(e, 1e-20) for e in errors], 'r^-', linewidth=2)
ax2.set_xlabel('w')
ax2.set_ylabel('|torch - Value| (log scale)')
ax2.set_title('Gradient error (atol level)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('max error atol:', max(errors))

## Wrap-up

`verify_gradients()` returned `dL/da=6.0, dL/db=7.0`, exactly matching the hand-derived analytic derivatives; the parameter experiment then reconfirmed that torch autograd and the hand-written `Value` engine agree to within 1e-6.
What this lesson really established: PyTorch `autograd`'s graph traversal, its storage of leaf gradients, and the `zero_grad()` convention are mathematically the same thing as the pure-Python implementation in `L56_backward_pass.ipynb`.
That equivalence is where the confidence to swap the hand-written engine for torch comes from — the training loop of the Aurora ML engine (`src/aurora/ml/`, a planned module) will call `loss.backward()` directly.
The next lesson (L61) hooks this machinery up to `nn.Module`: how `nn.Parameter` gets registered, and how `parameters()` / `state_dict()` manage a model.

> **retain_graph vs retain_grad**: `retain_grad()` keeps a non-leaf node's `.grad` value (demonstrated in this lesson); `loss.backward(retain_graph=True)` keeps the computation graph so `backward()` can be called several times (used in multi-task learning and similar setups — see section 3.5).

## ✏️ Whiteboard challenge: work through autograd by hand (target: 10 minutes)

Cover the screen and walk the graph with pen and paper first:

**Question 1**: `L = 2*a*b + 3*b`, `a=2.0, b=3.0`. Compute `∂L/∂a` and `∂L/∂b` by hand.
(∂L/∂a=2b=6.0, ∂L/∂b=2a+3=2×2+3=7.0)

**Question 2**: Why is a non-leaf node's `.grad` `None` by default? What does fixing that with `retain_grad()` cost you?
(It saves memory; keeping intermediate gradients increases memory use — do it for every intermediate node and you're storing a gradient for every activation)

**Question 3**: If you call `backward()` twice through the same graph (without `retain_graph=True`), what happens?
(The computation graph is freed after the first backward; the second one raises a RuntimeError)

**Question 4**: What's the difference between `w.grad.zero_()` and `w.grad = None`?
(zero_() keeps the grad Tensor object and fills it with zeros; grad=None drops the gradient reference; the latter is more common inside optimizers)

**Question 5**: If you compute with `x.detach()`, does the gradient still flow back to `x`? Why would you need detach?
(No, it doesn't flow back; detach cuts the gradient chain — commonly used when you pass logits into a loss computation but don't want them to affect certain parameters)

When you've finished the derivations, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import torch

# Q1: verify the hand-computed gradients
a_q = torch.tensor(2.0, requires_grad=True)
b_q = torch.tensor(3.0, requires_grad=True)
L_q = 2*a_q*b_q + 3*b_q
L_q.backward()
assert abs(a_q.grad.item() - 6.0) < 1e-6, f"a.grad={a_q.grad.item()}"
assert abs(b_q.grad.item() - 7.0) < 1e-6, f"b.grad={b_q.grad.item()}"
print(f"Q1 ✅  L=2ab+3b: ∂L/∂a={a_q.grad.item():.1f}=2b=6, ∂L/∂b={b_q.grad.item():.1f}=2a+3=7")

# Q2: non-leaf grad=None; retain_grad keeps it
x_q = torch.tensor(2.0, requires_grad=True)
y_q = x_q * 3    # non-leaf
y_q.retain_grad()
z_q = y_q ** 2
z_q.backward()
assert x_q.grad is not None      # the leaf has a grad
assert y_q.grad is not None      # retain_grad kept the intermediate grad
print(f"Q2 ✅  after retain_grad, non-leaf grad={y_q.grad.item():.1f}, leaf grad={x_q.grad.item():.1f}")

# Q3: a second backward raises
x2 = torch.tensor(1.0, requires_grad=True)
y2 = x2 ** 2
y2.backward()
try:
    y2.backward()
    print("Q3 ❌  should have raised, but didn't")
except RuntimeError as e:
    print(f"Q3 ✅  the second backward raised RuntimeError (graph already freed): {str(e)[:50]}...")

# Q4: zero_() vs grad=None
w_q = torch.tensor(1.0, requires_grad=True)
(w_q * 2).backward()
grad_obj_before = id(w_q.grad)
w_q.grad.zero_()
assert id(w_q.grad) == grad_obj_before   # same object
w_q.grad = None
assert w_q.grad is None
print(f"Q4 ✅  zero_() keeps the grad object; grad=None drops the reference")

# Q5: detach cuts the gradient
x5 = torch.tensor(3.0, requires_grad=True)
xd = x5.detach()
y5 = xd * 2
# y5 has no grad_fn
assert y5.grad_fn is None
print(f"Q5 ✅  after x.detach(): grad_fn of y=xd*2 is {y5.grad_fn} (None — the gradient won't flow back to x)")
print("\n🎉 Autograd whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l60_review = {
    "leaf_vs_nonleaf":         None,  # understand leaf vs non-leaf, and why non-leaf .grad=None? True/False
    "verify_gradients_impl":   None,  # verify_gradients() implemented correctly, gradient asserts pass? True/False
    "detach_understood":       None,  # understand when and why detach() cuts the gradient flow? True/False
    "grad_accumulation":       None,  # understand how gradients accumulate over repeated backward without clearing? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard derivations done by hand? True/False
}

unfilled = [k for k, v in l60_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l60_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L60 fully cleared! On to L61: nn.Module in practice')

In [ ]:
# Aurora connection: the split between inference and training
# aurora.llm.KVCache is a pure-numpy inference component — inference needs no gradients, so no computation graph (the from-scratch principle);
# the training side (src/aurora/ml/, a planned module) will rely directly on the torch autograd verified in this lesson.
from aurora.llm.kvcache import KVCache

print("aurora.llm.kvcache.KVCache:", KVCache)
print("KVCache base classes:", [b.__name__ for b in KVCache.__bases__], "→ pure numpy, no gradients, no computation graph")
print("The training side (the planned src/aurora/ml/) will use loss.backward() — exactly the mechanism verified in this lesson")

---

→ **Next lesson** · [L61 · `nn.Module` in practice](L61_pytorch_nn.ipynb)

> Next lesson covers **`nn.Module` in practice**: Linear / ReLU / Sequential, plus managing parameters and saving models.